# 🎓 Day 2 · Session 1
# 01B · Hands-on LoRA with FLAN-T5
## Train a small LoRA adapter and compare before/after behaviour

## 🎯 Learning Objectives

- Load a small model
- Apply LoRA
- Train on instruction examples
- Save the adapter
- Compare before and after outputs

## 🔧 Runtime Note

These notebooks work in VS Code/Jupyter.  
For hands-on LoRA training, **Google Colab with T4 GPU** is recommended.

```text
Runtime → Change runtime type → T4 GPU
```

# 1️⃣ Hands-on LoRA Fine-Tuning

This notebook demonstrates LoRA fine-tuning using a small model.

Recommended:
- Google Colab
- T4 GPU

We will:
1. Create a small instruction dataset
2. Load FLAN-T5-small
3. Apply LoRA
4. Train
5. Compare before and after
6. Save adapter

In [ ]:
# Install in Colab if required
# %pip install -q transformers datasets peft accelerate evaluate sentencepiece

In [ ]:
import torch
import pandas as pd

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 2️⃣ Tiny Teaching Dataset

In [ ]:
training_examples = [
    {"instruction": "Explain overfitting in machine learning.",
     "response": "Overfitting happens when a model learns the training data too closely, including noise. It performs well on training data but poorly on new data. Analogy: a student memorizes past papers but fails new questions."},
    {"instruction": "Explain underfitting in machine learning.",
     "response": "Underfitting happens when a model is too simple to learn the actual pattern. It performs poorly on both training and test data. Analogy: a student studies only chapter titles and cannot answer detailed questions."},
    {"instruction": "Explain train-test split.",
     "response": "Train-test split means dividing data into two parts: one to train the model and one to test it. This checks whether the model generalizes to unseen data."},
    {"instruction": "Explain regularization.",
     "response": "Regularization discourages a model from becoming too complex. It helps reduce overfitting and improves generalization."}
]
pd.DataFrame(training_examples)

# 3️⃣ Format Dataset

In [ ]:
def format_example(example):
    return f'''### Instruction:
{example["instruction"]}

### Response:
{example["response"]}'''

for ex in training_examples:
    print(format_example(ex))
    print("-" * 80)

# 4️⃣ Load Base Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Loaded:", model_name)

# 5️⃣ Test Before Training

In [ ]:
def generate_answer(model, prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    if torch.cuda.is_available():
        model = model.to("cuda")
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_answer(base_model, "Explain overfitting in machine learning."))

# 6️⃣ Apply LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# 7️⃣ Prepare Dataset

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list([
    {"input_text": ex["instruction"], "target_text": ex["response"]}
    for ex in training_examples
])

def preprocess(batch):
    inputs = tokenizer(batch["input_text"], max_length=128, truncation=True)
    labels = tokenizer(batch["target_text"], max_length=160, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_dataset = dataset.map(preprocess, batched=True)
tokenized_dataset

# 8️⃣ Train LoRA Adapter

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./lora_flan_t5_demo",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    learning_rate=5e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

# 9️⃣ Test After Training

In [ ]:
print(generate_answer(model, "Explain overfitting in machine learning."))
print()
print(generate_answer(model, "Explain regularization."))

# 🔟 Save Adapter

In [ ]:
adapter_path = "./flan_t5_lora_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print("Adapter saved to:", adapter_path)

# 🧪 Lab

Try:
1. Add 10 examples from your subject.
2. Change r=4, r=8, r=16.
3. Compare quality.
4. Test questions not in the training set.
5. Check if the model learned facts or style.